# K-Nearest Neighbors (KNN) in Jupyter Notebook
## Notes on KNN
What is KNN?

K-Nearest Neighbors (KNN) is a supervised machine learning algorithm used for both classification and regression tasks. In classification, it predicts the class of a new observation based on the classes of its nearest neighbors in the training data.

### How KNN works

#### For a new data point:

Compute its distance to all training observations.
Select the K nearest neighbors.
For classification, assign the class that appears most frequently among those neighbors.
Important idea

KNN does not build a traditional mathematical model during training. Instead, it stores the training data and makes predictions only when needed. Because of this, it is called a lazy learner.

### Key hyperparameters in KNN
n_neighbors (K): Number of neighbors to use.
weights:
"uniform": all neighbors contribute equally
"distance": closer neighbors have more influence
metric: distance measure used, such as:
* Euclidean
* Manhattan
* Minkowski

### Advantages of KNN
Simple to understand and implement
Works well for smaller datasets
No strong assumptions about data distribution
### Disadvantages of KNN
Can be slow on large datasets
Sensitive to irrelevant features
Sensitive to the scale of variables, so feature scaling is very important
Choice of K affects performance strongly
### Why scaling matters in KNN

KNN uses distances. If one feature has much larger values than others, it can dominate the distance calculation. That is why we usually standardize numerical features before fitting KNN.

# Example


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# ============================================
# 2. LOAD DATA
# ============================================
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/GkDzb7bWrtvGXdPOfk6CIg/Obesity-level-prediction-dataset.csv"
df = pd.read_csv(url)

# preview
print("Shape:", df.shape)
print(df.head())

Shape: (2111, 17)
   Gender   Age  Height  Weight family_history_with_overweight FAVC  FCVC  \
0  Female  21.0    1.62    64.0                            yes   no   2.0   
1  Female  21.0    1.52    56.0                            yes   no   3.0   
2    Male  23.0    1.80    77.0                            yes   no   2.0   
3    Male  27.0    1.80    87.0                             no   no   3.0   
4    Male  22.0    1.78    89.8                             no   no   2.0   

   NCP       CAEC SMOKE  CH2O  SCC  FAF  TUE        CALC  \
0  3.0  Sometimes    no   2.0   no  0.0  1.0          no   
1  3.0  Sometimes   yes   3.0  yes  3.0  0.0   Sometimes   
2  3.0  Sometimes    no   2.0   no  2.0  1.0  Frequently   
3  3.0  Sometimes    no   2.0   no  2.0  0.0  Frequently   
4  1.0  Sometimes    no   2.0   no  0.0  0.0   Sometimes   

                  MTRANS           NObeyesdad  
0  Public_Transportation        Normal_Weight  
1  Public_Transportation        Normal_Weight  
2  Public_Tran

In [3]:
X = df.drop("NObeyesdad", axis=1)
y = df["NObeyesdad"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (2111, 16)
Target shape: (2111,)


In [4]:
print("Unique target classes:", y.unique())

Unique target classes: <StringArray>
[      'Normal_Weight',  'Overweight_Level_I', 'Overweight_Level_II',
      'Obesity_Type_I', 'Insufficient_Weight',     'Obesity_Type_II',
    'Obesity_Type_III']
Length: 7, dtype: str


In [5]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Encoded classes:")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

Encoded classes:
0 -> Insufficient_Weight
1 -> Normal_Weight
2 -> Obesity_Type_I
3 -> Obesity_Type_II
4 -> Obesity_Type_III
5 -> Overweight_Level_I
6 -> Overweight_Level_II


In [6]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Categorical features: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


C:\Users\EmmanuelAdewuyi\AppData\Local\Temp\ipykernel_34068\1263164068.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1688, 16)
X_test shape: (423, 16)


In [8]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [9]:
print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Height', 'Weight', 'FCVC', 'NCP',
                                  'CH2O', 'FAF', 'TUE']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'family_history_with_overweight',
                                  'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC',
                                  'MTRANS'])])


In [10]:
baseline_knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

baseline_knn_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_knn_pipeline.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, y_pred_baseline))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_baseline, target_names=label_encoder.classes_))

Baseline Accuracy: 0.8250591016548463

Classification Report:

                     precision    recall  f1-score   support

Insufficient_Weight       0.78      0.94      0.86        54
      Normal_Weight       0.76      0.48      0.59        58
     Obesity_Type_I       0.74      0.96      0.84        70
    Obesity_Type_II       0.93      0.95      0.94        60
   Obesity_Type_III       0.98      1.00      0.99        65
 Overweight_Level_I       0.74      0.67      0.70        58
Overweight_Level_II       0.82      0.72      0.77        58

           accuracy                           0.83       423
          macro avg       0.82      0.82      0.81       423
       weighted avg       0.82      0.83      0.82       423



In [11]:
knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier())
])

param_grid = {
    "classifier__n_neighbors": [3, 5, 7, 9, 11, 13, 15],
    "classifier__weights": ["uniform", "distance"],
    "classifier__metric": ["euclidean", "manhattan", "minkowski"],
    "classifier__p": [1, 2]  # only relevant for Minkowski, but harmless here
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 84 candidates, totalling 420 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__metric': ['euclidean', 'manhattan', ...], 'classifier__n_neighbors': [3, 5, ...], 'classifier__p': [1, 2], 'classifier__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1

In [12]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest 5-Fold CV Accuracy:")
print(grid_search.best_score_)

Best Parameters:
{'classifier__metric': 'manhattan', 'classifier__n_neighbors': 3, 'classifier__p': 1, 'classifier__weights': 'distance'}

Best 5-Fold CV Accuracy:
0.8773708145312803


In [13]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Test Accuracy after tuning:", accuracy_score(y_test, y_pred_best))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best, target_names=label_encoder.classes_))

Test Accuracy after tuning: 0.8936170212765957

Classification Report:

                     precision    recall  f1-score   support

Insufficient_Weight       0.93      0.94      0.94        54
      Normal_Weight       0.74      0.64      0.69        58
     Obesity_Type_I       0.86      0.97      0.91        70
    Obesity_Type_II       1.00      0.98      0.99        60
   Obesity_Type_III       0.98      1.00      0.99        65
 Overweight_Level_I       0.82      0.79      0.81        58
Overweight_Level_II       0.90      0.90      0.90        58

           accuracy                           0.89       423
          macro avg       0.89      0.89      0.89       423
       weighted avg       0.89      0.89      0.89       423



In [14]:
cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print(cm_df)

                     Insufficient_Weight  Normal_Weight  Obesity_Type_I  \
Insufficient_Weight                   51              3               0   
Normal_Weight                          4             37               5   
Obesity_Type_I                         0              2              68   
Obesity_Type_II                        0              0               1   
Obesity_Type_III                       0              0               0   
Overweight_Level_I                     0              7               3   
Overweight_Level_II                    0              1               2   

                     Obesity_Type_II  Obesity_Type_III  Overweight_Level_I  \
Insufficient_Weight                0                 0                   0   
Normal_Weight                      0                 0                   8   
Obesity_Type_I                     0                 0                   0   
Obesity_Type_II                   59                 0                   0   
Obesity_T

In [15]:
baseline_acc = accuracy_score(y_test, y_pred_baseline)
tuned_acc = accuracy_score(y_test, y_pred_best)

print("Baseline Test Accuracy:", baseline_acc)
print("Tuned Test Accuracy:", tuned_acc)
print("Improvement:", tuned_acc - baseline_acc)

Baseline Test Accuracy: 0.8250591016548463
Tuned Test Accuracy: 0.8936170212765957
Improvement: 0.0685579196217494


In [17]:
results = pd.DataFrame(grid_search.cv_results_)
results = results.sort_values(by="mean_test_score", ascending=False)

print(results[[
    "params",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].head(10))

                                               params  mean_test_score  \
29  {'classifier__metric': 'manhattan', 'classifie...         0.877371   
57  {'classifier__metric': 'minkowski', 'classifie...         0.877371   
31  {'classifier__metric': 'manhattan', 'classifie...         0.877371   
33  {'classifier__metric': 'manhattan', 'classifie...         0.875599   
35  {'classifier__metric': 'manhattan', 'classifie...         0.875599   
61  {'classifier__metric': 'minkowski', 'classifie...         0.875599   
30  {'classifier__metric': 'manhattan', 'classifie...         0.872634   
28  {'classifier__metric': 'manhattan', 'classifie...         0.872634   
56  {'classifier__metric': 'minkowski', 'classifie...         0.872634   
37  {'classifier__metric': 'manhattan', 'classifie...         0.867900   

    std_test_score  rank_test_score  
29        0.015499                1  
57        0.015499                1  
31        0.015499                1  
33        0.013821               